# 2. Determining the cross-validation folds

| | |
| --- | --- |
| **Purpose** | Decide how to cut the data in time: a holdout that stays closed until the end, and walk-forward folds inside the rest |
| **Input** | The frozen universe copy saved by notebook 1 |
| **Output** | Decisions only. The rules are implemented by `find_holdout_start` and `find_fold_starts` in `src/wh_research/evaluation/splits.py`, so later notebooks recompute the same dates instead of copying them |
| **Next notebook** | `3_computing_labels_and_features` |
| **Workflow** | Step 4 of `docs/9-research_workflow.md` |

#### In previous notebook, we saved a copy of the universe dataset in folder "data/hyperliquid/case_study_data/universe_1d_min_mdv_2m_exit_1m_280323_171026.parquet"

It has every row of every symbol plus an `in_universe` column. The data we research on is the rows where `in_universe` is True.

In [15]:
from pathlib import Path
import pandas as pd

from wh_research.evaluation.splits import find_fold_starts, find_holdout_start

In [16]:
PROJECT_ROOT = Path.cwd().parent.parent
data_path = PROJECT_ROOT/"data/hyperliquid/case_study_data/universe_1d_min_mdv_2m_exit_1m_280323_171026.parquet"
df = pd.read_parquet(data_path)

#### Check length of data

Only the rows flagged in the universe count, so the period is from the first to the last day any coin was in the universe.

In [17]:
members = df[df["in_universe"]]
start_date = members["close_time"].min()
end_date = members["close_time"].max()

print(f"Start date: {str(start_date)}")
print(f"End date: {str(end_date)}")
print(f"Amount of data: {str(end_date-start_date)}")
print(f"Amount of data in years: {(end_date-start_date).days/365}")

Start date: 2023-03-28 23:59:59.999000+00:00
End date: 2026-10-17 23:59:59.999000+00:00
Amount of data: 1299 days 00:00:00
Amount of data in years: 3.558904109589041


#### We have 1299 days of data

We will want to split the data into two main sections:
- Train set
- Test set (holdout)

I want the holdout period to be around 1 year (365 days)\
The holdout period will be separated from the rest of the data and will not be opened until the final unbiased evaluation.

## Why walk-forward cross-validation and a holdout

- **Time series cannot be shuffled.** Neighbouring days are strongly related, and a label such as the 7-day forward return overlaps the next six days. A random k-fold split puts a day in training and its neighbours in evaluation, so the model has effectively seen the answer. Walk-forward cross-validation avoids this: every fold **trains on the past and evaluates on the future**.
- **The holdout is the final exam.** Everything we choose (features, labels, models, settings) is chosen on the train part. The holdout is looked at once, at the very end, so that the final number is an unbiased estimate. Every time you peek and adjust, the holdout loses that property.
- **Splits are cut by date.** The universe file is sorted by symbol first, so slicing by row position would cut by coin and not by time.
- **Proportions of days with symbols in the universe.** The cut points are fractions of the days on which at least one coin is in the universe, so empty days before the universe starts do not count.
- **Purging.** A label looks into the future, so a row near the end of a period has a label that reaches into the next period. Such rows are removed, see the conclusion and `docs/7-forward_returns.md`.

## Regular walk forward cross validation

In [18]:
# Proposed split of all data

# Train % of data, Holdout % of data = 70%, 30%

#
# Train    : ==============       (909.3 days)
# Holdout  :               ====== (389.7 days)
# Total    : ==================== (1299 days)

Within the Train set we will also split the data further into more folds

In [19]:
# Proposed split of the train set data

# if train data index = 0, 20
# fold 1 train : [:8]
# fold 1 eval  : [8:14]
# fold 2 train : [:14]
# fold 2 eval  : [14:]


# for the folds:
# = denotes data used for training
# + denotes data used for evaluation

# Fold 1    : ========++++++       (8 units of data used for training, 6 used for evaluation) (train: 363.72 days) (eval: 272.79 days)
# Fold 2    : ==============++++++ (14 units of data used for training, 6 used for evaluation) (train: 636.51 days) (eval: 272.79 days)
# Total     : ==================== (20 units total) (934 days)

## The rule as code

We have decided on (cut by date, as proportions of the days that have symbols in the universe):

- the **holdout** starts after the first 70% of the days (`find_holdout_start`)
- **fold 1** is evaluated from 40% of the train days, **fold 2** from 70% of the train days (`find_fold_starts`)

```
fold 1 train = open_time <  fold1_eval_start
fold 1 eval  = fold1_eval_start <= open_time < fold2_eval_start
fold 2 train = open_time <  fold2_eval_start
fold 2 eval  = fold2_eval_start <= open_time < holdout_start
```

The folds themselves are only built **after the features and the label are chosen** (notebook 4 onwards), because the purge depends on the label's horizon. Here we only fix the boundaries.

In [20]:
holdout_start = find_holdout_start(df)
fold1_eval_start, fold2_eval_start = find_fold_starts(df, holdout_start)

print(f"Fold 1 eval starts : {fold1_eval_start}")
print(f"Fold 2 eval starts : {fold2_eval_start}")
print(f"Holdout starts     : {holdout_start}")

Fold 1 eval starts : 2024-03-25 00:00:00+00:00
Fold 2 eval starts : 2024-12-23 00:00:00+00:00
Holdout starts     : 2025-09-22 00:00:00+00:00


#### The periods in days

How many days of data each period holds (days with at least one symbol in the universe, before any label purge).

In [21]:
days = df.loc[df["in_universe"], "open_time"].drop_duplicates().sort_values()

periods = {
    "fold 1 train (= block 1)": (days.min(), fold1_eval_start),
    "fold 1 eval (= block 2)": (fold1_eval_start, fold2_eval_start),
    "fold 2 eval (= block 3)": (fold2_eval_start, holdout_start),
    "holdout": (holdout_start, days.max() + pd.Timedelta(days=1)),
}

pd.DataFrame({
    name: {"from": start.date(), "to (not included)": end.date(), "days": int(((days >= start) & (days < end)).sum())}
    for name, (start, end) in periods.items()
}).T

,from,to (not included),days
fold 1 train (= block 1),2023-03-28,2024-03-25,363
fold 1 eval (= block 2),2024-03-25,2024-12-23,273
fold 2 eval (= block 3),2024-12-23,2025-09-22,273
holdout,2025-09-22,2026-10-18,391


# Conclusion

- We will perform all model research and development on the train-set, the holdout set is only for a final unbiased evaluation
- The splits are cut by **date**, not by row position. The file is sorted by symbol first, so slicing it by rows would cut by symbol instead of by time
- The labels look into the future, so a row whose label ends after the next period starts overlaps that period. The holdout split is purged with the longest label (7 days): keep the pre-holdout rows with `fwd_end_time_7 < holdout start`. The CV folds are built after the features and the label are chosen, and are purged with the chosen label's horizon: keep the training rows with `fwd_end_time_{n} < start of the evaluation period`

**What the next notebooks do with this:** notebook 3 adds the labels and features and cuts the holdout off. Notebook 4 evaluates the features on the pre-holdout data, using fold 1 train, fold 1 eval and fold 2 eval as three **non-overlapping blocks** (fold 2 train is block 1 plus block 2, so using it as well would count the same days twice).